# Intel Core Ultra NPU: GNN Benchmarking Suite

Research pipeline for evaluating Graph Neural Networks (GNNs) on Intel Core Ultra NPU (Meteor Lake) using OpenVINO.

## Overview
- **Real-world datasets**: OGBN-Arxiv, OGBN-Proteins, OGBN-Products
- **Statistical rigor**: 100+ iterations, warmup, multiple repeats
- **Profiling**: Operator-level execution, CPU fallback analysis, hardware metrics
- **Outputs**: PNG (300 DPI) + SVG for publication

## Research Questions
1. Are sparse GNN workloads memory-bound on consumer NPUs?
2. How does graph density affect NPU efficiency?
3. Do CNNs benefit more from operator fusion than GNNs?
4. Does CPU fallback impact transformer-based graph models?
5. Are current NPUs optimized for dense regular vs irregular sparse workloads?

In [ ]:
# Environment Setup
import os
import sys
import ctypes
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

is_admin = bool(ctypes.windll.shell32.IsUserAnAdmin())
print(f"Admin: {'Yes' if is_admin else 'No'}")

RESULTS_DIR = Path("results")
MODELS_DIR = Path("models")
FIGURES_DIR = Path("results/figures")

for d in [RESULTS_DIR, MODELS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.style.use('default')
sns.set_theme(style="whitegrid")
print("Environment ready.")

## Phase 1: Model Generation

In [ ]:
# Generate GNN and baseline models
!{sys.executable} analysis/model_prep.py

## Phase 2: Benchmarking

In [ ]:
# Main scalability benchmark
!{sys.executable} analysis/scalability_analyzer.py --iterations 100 --repeats 3 --profile --input-source auto --dataset-root data

In [ ]:
# Density sweep across datasets
!{sys.executable} analysis/density_sweep.py --models-dir models --results-dir results/density_sweep --datasets ogbn-arxiv,ogbn-proteins,ogbn-products --iterations 100 --repeats 3 --profile --dataset-root data --auto-models --gnn-nodes 4096 --devices CPU,NPU

In [ ]:
# Scaling analysis
!{sys.executable} analysis/scaling_sweep.py --dataset ogbn-arxiv --device NPU --dataset-root data --sizes 512,1024,2048,4096,8192 --model GCN --out-dir results/scaling_sweep --iterations 100 --repeats 3 --warmup 5

## Phase 3: Data Analysis

Aggregate benchmark results into analysis tables.

In [ ]:
# Aggregate density sweep results
from pathlib import Path
import json
import numpy as np
import pandas as pd
from analysis.plot_config import apply_ieee_style

apply_ieee_style()

sweep_dir = Path('results/density_sweep')
out_dir = Path('results/figures')
out_dir.mkdir(parents=True, exist_ok=True)

def _read_metadata(model_dir: Path):
    candidate = model_dir / 'run_00' / 'input_metadata.json'
    paths = [candidate] if candidate.exists() else sorted(model_dir.glob('run_*/input_metadata.json'))
    for p in paths:
        try:
            payload = json.loads(p.read_text(encoding='utf-8'))
            return {k: float(payload.get(k)) for k in ('used_num_nodes', 'used_num_edges', 'used_num_features') if isinstance(payload.get(k), (int, float))}
        except:
            continue
    return {}

rows = []
for ds_dir in sorted(sweep_dir.glob('dataset_*')):
    if not ds_dir.is_dir():
        continue
    matrix = ds_dir / 'scalability_matrix.csv'
    if not matrix.exists():
        continue
    df_ds = pd.read_csv(matrix)
    df_ds['dataset'] = ds_dir.name.replace('dataset_', '')
    meta_list = [_read_metadata(ds_dir / model) for model in df_ds['model'].astype(str)]
    df_ds = pd.concat([df_ds, pd.DataFrame(meta_list)], axis=1)
    if {'used_num_nodes', 'used_num_edges'}.issubset(df_ds.columns):
        df_ds['edges_per_node'] = df_ds['used_num_edges'] / df_ds['used_num_nodes'].replace(0, np.nan)
    rows.append(df_ds)

if rows:
    density_df = pd.concat(rows, ignore_index=True)
    density_df.to_csv(out_dir / 'density_sweep_merged.csv', index=False)
    summary = density_df.groupby('dataset').agg(edges_per_node=('edges_per_node', 'mean'), latency_ms=('o_mean_ms', 'mean')).sort_values('edges_per_node')
    summary.to_csv(out_dir / 'density_summary_by_dataset.csv')
    print(f"Density analysis: {len(density_df)} records")
    print(summary)
else:
    print("No density data found")

In [ ]:
# Analyze graph topology
from analysis.graph_topology_analyzer import GraphTopologyAnalyzer

analyzer = GraphTopologyAnalyzer(results_dir=out_dir, dataset_root=Path('data'))
stats_df = analyzer.analyze_datasets(['ogbn-arxiv', 'ogbn-proteins', 'ogbn-products'])
print("\nDataset Statistics:")
print(stats_df[['dataset_name', 'num_nodes', 'num_edges', 'avg_degree', 'density', 'power_law_alpha']].to_string(index=False))

In [ ]:
# Analyze operator composition
import onnx
from analysis.plot_config import get_model_category

categories = ['SpMM/MatMul', 'MLP', 'Activation', 'Attention', 'Memory/Shape', 'Other']

def categorize(op_type):
    op = str(op_type)
    if op in {'MatMul', 'Gemm'}:
        return 'SpMM/MatMul'
    if op in {'Conv', 'ConvTranspose', 'DepthwiseConv'}:
        return 'MLP'
    if 'Attention' in op or op in {'Softmax', 'LayerNormalization'}:
        return 'Attention'
    if op in {'Relu', 'Sigmoid', 'Tanh', 'Gelu'}:
        return 'Activation'
    if op in {'Gather', 'Scatter', 'Slice', 'Concat', 'Transpose', 'Reshape'}:
        return 'Memory/Shape'
    return 'Other'

rows = []
for model_path in sorted(Path('models').glob('*.onnx')):
    if 'fp32' not in model_path.stem.lower():
        continue
    try:
        model = onnx.load(str(model_path))
        counts = {c: 0 for c in categories}
        for node in model.graph.node:
            counts[categorize(node.op_type)] += 1
        total = sum(counts.values()) or 1
        row = {k: (v/total)*100 for k, v in counts.items()}
        row['model'] = model_path.stem
        row['category'] = get_model_category(model_path.stem)
        rows.append(row)
    except Exception as e:
        print(f"Error: {e}")

if rows:
    op_df = pd.DataFrame(rows).sort_values(['category', 'model'])
    op_df.to_csv(out_dir / 'operator_mix.csv', index=False)
    print(f"Operator analysis: {len(op_df)} models")
else:
    print("No operator data")

## Phase 4: Visualizations

One figure per cell. Each generates PNG and SVG outputs.

In [ ]:
# Figure 1: Density vs Performance
from analysis.plot_config import apply_ieee_style, IEEE_COLORS, save_figure_dual_format

apply_ieee_style()

fig, ax = plt.subplots(figsize=(3.5, 2.5))

if 'density_df' in locals() and not density_df.empty:
    npu_data = density_df[density_df.get('device', 'NPU') == 'NPU']
    for i, (dataset, group) in enumerate(npu_data.groupby('dataset')):
        if group.get('edges_per_node').notna().any():
            x = group['edges_per_node'].mean()
            y = group['o_mean_ms'].mean()
            ax.scatter(x, y, s=80, color=IEEE_COLORS[i % len(IEEE_COLORS)], label=dataset, edgecolors='black', linewidth=0.5)

    ax.set_xlabel('Average Degree (edges/node)')
    ax.set_ylabel('Latency (ms)')
    ax.set_title('Figure 1: Density vs Performance')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

save_figure_dual_format(fig, out_dir / 'fig1_density_vs_performance')
display(Image(filename=out_dir / 'fig1_density_vs_performance.png'))

In [ ]:
# Figure 2: Degree Distribution (Log-Log)
from analysis.plot_config import save_figure_dual_format

if Path(out_dir / 'fig2_degree_distribution_loglog.png').exists():
    display(Image(filename=out_dir / 'fig2_degree_distribution_loglog.png'))
else:
    print("Figure not generated. Check graph topology analyzer.")

In [ ]:
# Figure 3: Operator Breakdown
import numpy as np

if 'op_df' in locals() and not op_df.empty:
    fig, ax = plt.subplots(figsize=(7.16, 3))
    x = np.arange(len(op_df))
    bottom = np.zeros(len(op_df))
    colors = {cat: IEEE_COLORS[i % len(IEEE_COLORS)] for i, cat in enumerate(categories)}

    for cat in categories:
        vals = op_df[cat].fillna(0).values
        ax.bar(x, vals, bottom=bottom, label=cat, color=colors[cat], edgecolor='k', linewidth=0.25)
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(op_df['model'], rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Share of ONNX nodes (%)')
    ax.set_title('Figure 3: Operator Breakdown')
    ax.legend(ncol=3, fontsize=7)
    ax.set_ylim(0, 120)

    save_figure_dual_format(fig, out_dir / 'fig3_operator_breakdown')
    display(Image(filename=out_dir / 'fig3_operator_breakdown.png'))
else:
    print("No operator data available")

In [ ]:
# Figure 4: CPU Fallback Heatmap
from analysis.ort_profile_utils import iter_operator_events, load_events
from analysis.plot_config import shorten_label

results_dir = Path('results')
mode = 'optimized'

model_maps = {}
op_total_time = {}

for model_dir in [p for p in results_dir.iterdir() if p.is_dir()]:
    traces = sorted(model_dir.glob(f'run_*/{mode}_profiling.json'))
    if not traces:
        continue
    try:
        events = load_events(traces[-1])
        sums = {}
        for op_name, dur_us, provider, _ in iter_operator_events(events):
            op = str(op_name)
            if op not in sums:
                sums[op] = {'cpu': 0, 'total': 0}
            sums[op]['total'] += dur_us
            if 'cpu' in str(provider).lower():
                sums[op]['cpu'] += dur_us
            op_total_time[op] = op_total_time.get(op, 0) + dur_us
        cpu_frac = {op: v['cpu']/v['total'] if v['total'] > 0 else 0 for op, v in sums.items()}
        model_maps[model_dir.name] = cpu_frac
    except:
        pass

if model_maps:
    top_ops = [op for op, _ in sorted(op_total_time.items(), key=lambda x: x[1], reverse=True)[:25]]
    sorted_models = sorted(model_maps.keys())
    mat = np.full((len(top_ops), len(sorted_models)), np.nan)
    for j, model in enumerate(sorted_models):
        mp = model_maps[model]
        for i, op in enumerate(top_ops):
            if op in mp:
                mat[i, j] = mp[op]

    fig, ax = plt.subplots(figsize=(7.16, 4))
    im = ax.imshow(mat, aspect='auto', vmin=0, vmax=1, cmap='RdYlGn_r')
    ax.set_xticks(range(len(sorted_models)))
    ax.set_xticklabels([shorten_label(m, 14) for m in sorted_models], rotation=45, ha='right')
    ax.set_yticks(range(len(top_ops)))
    ax.set_yticklabels([shorten_label(op, 30) for op in top_ops])
    ax.set_xlabel('Model')
    ax.set_ylabel('Operator')
    ax.set_title('Figure 4: CPU Fallback Heatmap')
    fig.colorbar(im, ax=ax, label='CPU fraction')

    save_figure_dual_format(fig, out_dir / 'fig4_cpu_fallback_heatmap')
    display(Image(filename=out_dir / 'fig4_cpu_fallback_heatmap.png'))
else:
    print("No profiling data found")

In [ ]:
# Figure 5: Fusion Gain vs Performance
from scipy import stats

matrix_csv = Path('results/scalability_matrix.csv')

if matrix_csv.exists():
    df = pd.read_csv(matrix_csv)
    df['lat_impr_pct'] = (df['b_mean_ms'] - df['o_mean_ms']) / df['b_mean_ms'].replace(0, np.nan) * 100
    df['category'] = df['model'].apply(get_model_category)

    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    markers = {'GNN (Irregular)': 'o', 'CNN (Regular)': 's', 'Transformer (Global Attn)': '^'}

    for cat in df['category'].unique():
        cat_data = df[df['category'] == cat]
        ax.scatter(cat_data['speedup'], cat_data['lat_impr_pct'],
                  s=50, alpha=0.8, label=cat, marker=markers.get(cat, 'o'), edgecolors='black', linewidth=0.5)

    ax.axvline(1.0, color='black', linestyle='--', linewidth=0.8)
    ax.axhline(0.0, color='gray', linestyle=':', linewidth=0.6)

    valid = df[df['speedup'].notna() & df['lat_impr_pct'].notna()]
    if len(valid) > 3:
        r, p = stats.pearsonr(valid['speedup'], valid['lat_impr_pct'])
        ax.text(0.05, 0.95, f'r={r:.2f}, p={p:.3f}', transform=ax.transAxes,
               fontsize=8, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

    ax.set_xlabel('Fusion Gain Ratio')
    ax.set_ylabel('Latency improvement (%)')
    ax.set_title('Figure 5: Fusion Gain vs Performance')
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

    save_figure_dual_format(fig, out_dir / 'fig5_fusion_gain')
    display(Image(filename=out_dir / 'fig5_fusion_gain.png'))
else:
    print("Scalability matrix not found")

In [ ]:
# Figure 6: Scaling Analysis
scaling_csv = Path('results/scaling_sweep/scaling_sweep.csv')

if scaling_csv.exists():
    df = pd.read_csv(scaling_csv)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.16, 3))

    # Node scaling
    ax1.plot(df['num_nodes'], df['o_mean_ms'], 'o-', color=IEEE_COLORS[0], linewidth=1.5, markersize=6)
    ax1.set_xlabel('Number of Nodes')
    ax1.set_ylabel('Latency (ms)')
    ax1.set_title('Scaling by Nodes')
    ax1.grid(True, alpha=0.3)

    # Edge scaling
    ax2.plot(df['num_edges'], df['o_mean_ms'], 's-', color=IEEE_COLORS[1], linewidth=1.5, markersize=6)
    ax2.set_xlabel('Number of Edges')
    ax2.set_ylabel('Latency (ms)')
    ax2.set_title('Scaling by Edges')
    ax2.grid(True, alpha=0.3)

    fig.suptitle('Figure 6: Scaling Characteristics', fontsize=10)
    plt.tight_layout()

    save_figure_dual_format(fig, out_dir / 'fig6_scaling')
    display(Image(filename=out_dir / 'fig6_scaling.png'))
else:
    print("Scaling data not found")

## Summary

Generated outputs in `results/figures/`:
- PNG (300 DPI) for publications
- SVG for vector editing

In [ ]:
# List all generated outputs
print("Generated Files:")
for f in sorted(out_dir.glob('fig*.png')):
    print(f"  {f.name}")
for f in sorted(out_dir.glob('fig*.svg')):
    print(f"  {f.name}")
for f in sorted(out_dir.glob('*.csv')):
    print(f"  {f.name}")